# Config

In [1]:
!pip install nltk==3.9.1
!pip install mlflow==3.3.1

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [2]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [3]:
import pandas as pd
import os
from preprocess.preprocess import clean_text, check_deleted_expressions
from preprocess.translate import translator, gen_text_for_embedding, final_clean, detect_language
import time
import json

# 1) RoBERTa test

In [ ]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [15]:
#del gen_dataset
from utils.dataset import gen_dataset
from models.specter import embed_texts
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [ ]:
# -------------------------------
# Fine-tuning de RoBERTa en una lista de textos
# -------------------------------
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
from torch.optim import AdamW
from transformers import get_scheduler
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ===========================
# 1. Datos de ejemplo
# ===========================
"""
texts = [
    "El producto llegó en buen estado.",
    "El envío fue demasiado lento.",
    "Excelente calidad, lo recomiendo.",
    "Muy mala atención al cliente."
]
labels = [1, 0, 1, 0]   # 1 = positivo, 0 = negativo

# Dividir train/test
X_train, X_val, y_train, y_val = train_test_split(texts, labels, test_size=0.2, random_state=42)
"""
# ===========================
# 2. Tokenizador
# ===========================
model_name = "roberta-base"
tokenizer = RobertaTokenizerFast.from_pretrained(model_name)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = TextDataset(X_train, y_train, tokenizer)
test_dataset   = TextDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader   = DataLoader(test_dataset, batch_size=2)

# ===========================
# 3. Modelo
# ===========================
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ===========================
# 4. Optimizador y scheduler
# ===========================
optimizer = AdamW(model.parameters(), lr=1e-3)
num_training_steps = len(train_loader) * 3  # 3 epochs
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# ===========================
# 5. Entrenamiento
# ===========================
epochs = 10

model.train()
for epoch in tqdm(range(epochs)):
    model.train()
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        

    # ===========================
    # 6. Evaluación rápida
    # ===========================
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            preds = torch.argmax(outputs.logits, dim=-1)
            correct += (preds == batch["labels"]).sum().item()
            total += len(batch["labels"])



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
 10%|█         | 1/10 [00:16<02:31, 16.81s/it]

Accuracy en validación: 0.42


 20%|██        | 2/10 [00:33<02:14, 16.81s/it]

Accuracy en validación: 0.58


 30%|███       | 3/10 [00:50<01:57, 16.74s/it]

...

Accuracy en validación: 0.58


 90%|█████████ | 9/10 [02:30<00:16, 16.72s/it]

Accuracy en validación: 0.58


100%|██████████| 10/10 [02:47<00:00, 16.72s/it]

Accuracy en validación: 0.58


# 2) SPECTER test

In [4]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [5]:
#del gen_dataset
from utils.dataset import gen_dataset
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

## Sin criterion

In [ ]:
# Fine-tuning de SPECTER (allenai/specter) para clasificación
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from tqdm import tqdm

"""
# ---------- Datos de ejemplo ----------
texts = [
    "Deep learning improves medical image classification.",
    "We propose a new finite element method for biomechanics.",
    "A survey on NLP for legal documents.",
    "Spectral methods for time series forecasting."
]

labels = [1, 1, 0, 0]  # ej. 2 clases
X_train, X_val, y_train, y_val = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)
"""
# Si tienes título y abstract, arma algo así:
# texts = [f"{title} [SEP] {abstract}" for title, abstract in data]

# ---------- Tokenizador y Dataset ----------
model_name = "allenai/specter"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.enc = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = TextDataset(X_train, y_train, tokenizer)
val_ds   = TextDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False)

# ---------- Modelo (capa de clasificación encima de SPECTER) ----------
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ---------- Optimizador y scheduler ----------
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
epochs = 10
num_training_steps = len(train_loader) * epochs
scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

# ---------- Loop de entrenamiento ----------
for epoch in range(epochs):
    model.train()
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for batch in loop:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(**batch)
        loss = out.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        loop.set_postfix(loss=loss.item())

    # ---------- Evaluación rápida ----------
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            preds = out.logits.argmax(dim=-1)
            correct += (preds == batch["labels"]).sum().item()
            total += batch["labels"].size(0)
    print(f"Val acc: {correct/total:.3f}")

# (Opcional) Guardar modelo
# model.save_pretrained("./specter_cls")
# tokenizer.save_pretrained("./specter_cls")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/specter and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1/10: 100%|██████████| 97/97 [00:12<00:00,  7.66it/s, loss=1.07] 


Val acc: 0.648


Epoch 2/10: 100%|██████████| 97/97 [00:12<00:00,  7.66it/s, loss=0.469]


Val acc: 0.653


Epoch 3/10: 100%|██████████| 97/97 [00:12<00:00,  7.60it/s, loss=0.0763]


...

Val acc: 0.710


Epoch 9/10: 100%|██████████| 97/97 [00:12<00:00,  7.51it/s, loss=0.00525]


Val acc: 0.710


Epoch 10/10: 100%|██████████| 97/97 [00:12<00:00,  7.54it/s, loss=0.00167]


Val acc: 0.710


## Con criterion

In [6]:
# Fine-tuning de SPECTER (allenai/specter) para clasificación
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from tqdm import tqdm

def get_sample_weights_loss(y):
  y = np.asarray(y, dtype=np.int64)
  class_counts = np.bincount(y)
  class_weights = 1.0 / class_counts
  class_weights = class_weights / class_weights.sum()

  return class_weights

"""
# ---------- Datos de ejemplo ----------
texts = [
    "Deep learning improves medical image classification.",
    "We propose a new finite element method for biomechanics.",
    "A survey on NLP for legal documents.",
    "Spectral methods for time series forecasting."
]

labels = [1, 1, 0, 0]  # ej. 2 clases
X_train, X_val, y_train, y_val = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)
"""
# Si tienes título y abstract, arma algo así:
# texts = [f"{title} [SEP] {abstract}" for title, abstract in data]

# ---------- Tokenizador y Dataset ----------
model_name = "allenai/specter"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.enc = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = TextDataset(X_train, y_train, tokenizer)
val_ds   = TextDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False)

# ---------- Modelo (capa de clasificación encima de SPECTER) ----------
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ---------- Optimizador y scheduler ----------
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
epochs = 10
num_training_steps = len(train_loader) * epochs
scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)
class_weights = get_sample_weights_loss(y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

# ---------- Loop de entrenamiento ----------
for epoch in range(epochs):
    model.train()
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for batch in loop:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(**{k: v for k, v in batch.items() if k != "labels"})
        logits = out.logits
        loss = criterion(logits, batch["labels"].to(device))
        loss.backward()
        optimizer.step()
        scheduler.step()
        loop.set_postfix(loss=loss.item())

    # ---------- Evaluación rápida ----------
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            preds = out.logits.argmax(dim=-1)
            correct += (preds == batch["labels"]).sum().item()
            total += batch["labels"].size(0)
    print(f"Val acc: {correct/total:.3f}")


/usr/local/lib/python3.10/dist-packages/torch/_utils.py:830: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/specter and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1/10: 100%|██████████| 193/193 [00:14<00:00, 12.92it/s, loss=0.651]


Val acc: 0.663


Epoch 2/10: 100%|██████████| 193/193 [00:14<00:00, 13.12it/s, loss=0.312]


Val acc: 0.663


Epoch 3/10: 100%|██████████| 193/193 [00:14<00:00, 13.15it/s, loss=0.144] 


...

Val acc: 0.731


Epoch 9/10: 100%|██████████| 193/193 [00:14<00:00, 13.01it/s, loss=0.00102] 


Val acc: 0.715


Epoch 10/10: 100%|██████████| 193/193 [00:14<00:00, 12.89it/s, loss=0.00154] 


Val acc: 0.715


# 3) SPECTER

## Functions

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from torch.optim import AdamW


class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.enc = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

model_name = "allenai/specter"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_ds = TextDataset(X_train, y_train, tokenizer)
val_ds   = TextDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False)

# ---------- Modelo (capa de clasificación encima de SPECTER) ----------
model = AutoModelForSequenceClassification.from_pretrained(model_name)
    

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import os
import numpy as np
from collections import Counter
import copy
import inspect

def get_sample_weights_loss(y):
  y = np.asarray(y, dtype=np.int64)
  class_counts = np.bincount(y)
  class_weights = 1.0 / class_counts
  class_weights = class_weights / class_weights.sum()

  return class_weights


class Pytorch_Pipeline():
      def __init__(self, model_class, sample_weights_loss=None, max_epochs = 200, use_scheduler=None):
        #Set device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        #Modelo
        self.model_class = model_class
        self.model = None
        #Elementos del entrenamiento
        self.params = None
        self.sample_weights_loss = sample_weights_loss
        self.criterion = None
        self.optimizer = None
        self.batch_size = None
        self.max_epochs = max_epochs
        #scheduler
        self.use_scheduler=use_scheduler

      def partial_fit(self, loader):
        self.model.to(self.device)
        self.model.train()
        if self.use_scheduler is not None:
            scheduler = get_scheduler(
                "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
            )

        for xb, yb in loader:
            xb, yb = xb.to(self.device), yb.to(self.device)
            self.optimizer.zero_grad()
            out = self.model(xb)
            logits = out.logits
            loss = self.criterion(logits, batch["labels"].to(device))
            loss.backward()
            self.optimizer.step()

        return self
        
      def predict_and_evaluate(self, loader):
          self.model.eval()
          val_loss, n_samples = 0.0, 0
          all_preds, all_targets = [], []
          with torch.no_grad():
              for xb, yb in loader:
                  xb, yb = xb.to(self.device), yb.to(self.device)
                  output = self.model(xb)
                  loss = self.criterion(output, yb)
                  val_loss += self.criterion(output, yb).item() * xb.size(0)
                  n_samples += xb.size(0)

                  pred = output.argmax(dim=1)
                  all_preds.append(pred.cpu())
                  all_targets.append(yb.cpu())

          avg_val_loss = val_loss / n_samples
          y_true = torch.cat(all_targets).numpy()
          y_pred = torch.cat(all_preds).numpy()
          f1 = f1_score(y_true, y_pred, average='weighted')  # weighted F1

          return avg_val_loss, f1, y_true, y_pred

      def set_params(self, **params):
          self.params = params

          # Obtener los parámetros esperados por el constructor de model_class
          signature = inspect.signature(self.model_class.__init__)
          valid_keys = set(signature.parameters.keys()) - {'self'}

          # Filtrar los params para incluir solo los esperados
          filtered_params = {k: v for k, v in params.items() if k in valid_keys}

          self.model = self.model_class(**filtered_params)
          self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.params['lr'])
          self.batch_size = self.params['batch_size']

      def set_criterion(self, y):
          # ----------- Criterion -----------
          if self.sample_weights_loss is not None:
              class_weights = get_sample_weights_loss(y)
              class_weights = torch.tensor(class_weights, dtype=torch.float32).to(self.device)
              self.criterion = nn.CrossEntropyLoss(weight=class_weights)
          else:
              self.criterion = nn.CrossEntropyLoss()

          return self

      def fit_early_stopping(self, train_loader, test_loader, labels):

          self.set_criterion(labels)

          for epoch in range(self.max_epochs):
              self.partial_fit(train_loader)
              avg_val_loss, f1, _, _ = self.predict_and_evaluate(test_loader)
              # ---------- Early stopping (por pérdida) ----------
              patience = 10
              min_delta = 1e-4
              best_val_loss = float('inf')
              epochs_no_improve = 0
              best_model_state = None

              if avg_val_loss + min_delta < best_val_loss:
                  best_val_loss = avg_val_loss
                  best_model_state = self.model.state_dict()
                  epochs_no_improve = 0
              else:
                  epochs_no_improve += 1
                  if epochs_no_improve >= patience:
                      break

          return f1



## Code

In [ ]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)